# Philippine Rice Price Forecasting — Updated Models

This notebook evaluates monthly Philippine rice prices using classical machine-learning regression models. It excludes Naive Bayes and SARIMA.

**Models:** Naive Forecast baseline, Random Forest Regressor, Extra Trees Regressor, Ridge Regression, and XGBoost Regressor.

**Data:** `rice_prices_cleaned.csv`, in long format with `Region`, `Year`, `Month`, `Rice_Type`, and `Price_PHP_per_kg`. Incomplete future months after September 2026 are excluded by the forecasting module.

> The Naive Forecast is a baseline that repeats the last observed price. It is not a trained regression model. Model scores must be verified by running this notebook on your dataset.

## 1. Install dependencies

In [ ]:
%pip -q install pandas numpy scikit-learn xgboost matplotlib

## 2. Load the dataset

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path("rice_prices_cleaned.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Place rice_prices_cleaned.csv in the same folder as this notebook.")
raw_prices = pd.read_csv(DATA_PATH)
print("Dataset shape:", raw_prices.shape)
print("Columns:", raw_prices.columns.tolist())
display(raw_prices.head())

## 3. Load the forecasting functions

In [ ]:
# Keep rice_forecasting.py in the same folder as this notebook.
from rice_forecasting import load_prices, make_supervised, evaluate_models, forecast_future, MODEL_NAMES

prices = load_prices(raw_prices)
print("Date range:", prices["Date"].min().date(), "to", prices["Date"].max().date())
print("Regions/geographies:", prices["Region"].nunique())
print("Rice types:", sorted(prices["Rice_Type"].unique()))
display(prices.head())

## 4. Explore historical prices

In [ ]:
monthly = prices.groupby(["Date", "Rice_Type"], as_index=False)["Price_PHP_per_kg"].mean()
for rice_type, group in monthly.groupby("Rice_Type"):
    plt.figure(figsize=(12, 4))
    plt.plot(group["Date"], group["Price_PHP_per_kg"], marker=".")
    plt.title(f"Average historical price — {rice_type}")
    plt.xlabel("Month")
    plt.ylabel("Price (PHP/kg)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Create supervised features

In [ ]:
supervised = make_supervised(prices)
print("Supervised rows:", len(supervised))
print("Target date range:", supervised["target_date"].min().date(), "to", supervised["target_date"].max().date())
display(supervised.head())

## 6. Train and compare the models

In [ ]:
# The latest 12 target months are held out for testing; rows are not randomly shuffled.
scores, predictions, details = evaluate_models(prices, test_months=12, include_xgboost=True)
print("Test period starts:", pd.Timestamp(details["cutoff"]).date())
display(scores.round(3))

### Understanding the metrics
- **MAE:** average absolute error in PHP/kg. Lower is better.
- **RMSE:** penalizes large errors more heavily. Lower is better.
- **MAPE:** average absolute percentage error. Lower is better, but interpret cautiously.

The Naive Forecast is a benchmark. A strong score alone does not guarantee a changing or realistic multi-month forecast.

## 7. Compare actual and predicted prices on the test set

In [ ]:
# Select a model from the results table. Change this value if desired.
model_to_plot = scores.iloc[0]["Model"]
plot_df = predictions[predictions["Model"] == model_to_plot].copy()
plot_df["target_date"] = pd.to_datetime(plot_df["target_date"])
plot_df = plot_df.sort_values("target_date")

plt.figure(figsize=(12, 5))
plt.plot(plot_df["target_date"], plot_df["target_price"], label="Actual", linewidth=2)
plt.plot(plot_df["target_date"], plot_df["Predicted_Price"], label=f"Predicted — {model_to_plot}", linewidth=2)
plt.title(f"Actual vs predicted prices — {model_to_plot}")
plt.xlabel("Target month")
plt.ylabel("Price (PHP/kg)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Forecast the next six months

In [ ]:
# Choose a region and rice type from the available data.
region = sorted(prices["Region"].unique())[0]
rice_type = sorted(prices["Rice_Type"].unique())[0]
model_kind = "Random Forest"  # Options: Naive Forecast, Random Forest, Extra Trees, Ridge Regression, XGBoost

forecast, fitted_model = forecast_future(
    prices,
    region=region,
    rice_type=rice_type,
    months=6,
    model_kind=model_kind,
)
print("Region:", region)
print("Rice type:", rice_type)
print("Model:", model_kind)
display(forecast.round(2))

## 9. Visualize the future forecast

In [ ]:
history = prices[(prices["Region"] == region) & (prices["Rice_Type"] == rice_type)].sort_values("Date")
plt.figure(figsize=(12, 5))
plt.plot(history["Date"].tail(24), history["Price_PHP_per_kg"].tail(24), label="Historical", marker="o")
plt.plot(forecast["Date"], forecast["Forecast_Price_PHP_per_kg"], label=f"Forecast — {model_kind}", marker="o")
plt.title(f"Rice price forecast — {rice_type}, {region}")
plt.xlabel("Month")
plt.ylabel("Price (PHP/kg)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Note:** If `model_kind = "Naive Forecast"`, a flat line is expected because the baseline repeats the latest observed price. The regression models produce recursive predictions, where each forecasted month becomes input for later months. Forecasts can still be flat or smooth if that is what the model predicts; do not force values to change artificially.

## 10. Save evaluation and forecast outputs

In [ ]:
scores.to_csv("rice_model_metrics.csv", index=False)
predictions.to_csv("rice_test_predictions.csv", index=False)
forecast.to_csv("rice_future_forecast.csv", index=False)
print("Saved: rice_model_metrics.csv, rice_test_predictions.csv, rice_future_forecast.csv")

# In Google Colab, uncomment these lines to download the CSV files:
# from google.colab import files
# files.download("rice_model_metrics.csv")
# files.download("rice_test_predictions.csv")
# files.download("rice_future_forecast.csv")

## 11. Interpretation and limitations

1. Identify the model with the lowest test MAE and RMSE.
2. Check whether each regression model improves on the Naive Forecast baseline.
3. Inspect actual-versus-predicted plots, not only the metric table.
4. Discuss differences across regions and rice types where relevant.
5. Explain that future estimates are uncertain. Weather, inflation, fuel costs, imports, and supply changes may affect prices but are not included unless reliable data are added.

**Important:** Run every cell from top to bottom. Keep `rice_forecasting.py` and `rice_prices_cleaned.csv` in the same folder as this notebook.